# Smart-meter Data Exploration

## Imports

In [30]:
import os
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from IPython.display import clear_output

## Load data

### Directories and filepaths

After more processed data has been generated, add filepaths here.

In [14]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')

### Load data

After more processed data has been generated, load it here.

In [15]:
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

## Prepare Data

In [41]:
df = pd.merge(ausgrid_df, open_meteo_df, on="datetime", how="inner")

# Get customers
customers = sorted(df['Customer'].dropna().unique())

# Get available numberic variables
variables = [
    col for col in df.columns
    if col not in ["datetime", "date", "time", "Customer"]
    and pd.api.types.is_numeric_dtype(df[col])]

#### Construct day-of-year and hour-of-day numeric columns

In [42]:
df['date'] = df['datetime'].dt.date
df['hour'] = df["datetime"].dt.hour
df["day_of_year"] = df["datetime"].dt.dayofyear  # 1–365 (or 366 in leap years)
df["hour_of_day"] = df["datetime"].dt.hour              # 0–23

## Daily Profile Dashboard

### Construct interactive controls

In [ ]:
# Customer dropdown
customer_dropdown = widgets.Dropdown(
    options=customers,
    description='Customer',
    layout=widgets.Layout(width='400px'))

# Date dropdown
date_dropdown = widgets.Dropdown(
    description='Date:',
    layout=widgets.Layout(width='400px'))

# Variable dropdown
variable_dropdown = widgets.Dropdown(
    options=variables,
    description='Variable:',
    layout=widgets.Layout(width='400px'))

# Normalize toggle
normalize_toggle = widgets.ToggleButton(
    value=False,
    description='Normalize: OFF',
    button_style='',
    layout=widgets.Layout(width="150px"))

# Add profile button
add_button = widgets.Button(
    description='Add profile',
    button_style='primary',
    layout=widgets.Layout(width='150px'))

# Clear profile button
clear_button = widgets.Button(
    description='Clear profiles',
    layout=widgets.Layout(width='150px'))

output = widgets.Output()

### Define functions for updating the dashboard

#### Update dates
Update the available dates when the customer changes.

In [44]:
def update_dates(change):
    customer = change['new']
    customer_df = df[df['Customer'] == customer]
    dates = sorted(customer_df['date'].unique())
    
    date_dropdown.options = dates
    if dates: date_dropdown.value = dates[0]

#### Update toggle label

In [45]:
def update_normalize_button(change):
    if change['new']:
        normalize_toggle.description = 'Normalize: ON'
        normalize_toggle.button_style = 'success'
    else:
        normalize_toggle.description = 'Normalize: OFF'
        normalize_toggle.button_style = ''

### Update plot

Draw all currently selected profiles

In [46]:
def update_plot(change=None):

    with output:
        clear_output(wait=True)
        plt.figure(figsize=(10, 5))

        # Plot every profile currently stored
        for customer, date, variable, normalized in profiles:
            day_df = df[(df['Customer'] == customer) & (df['date'] == date)].sort_values('datetime')
            if day_df.empty: continue

            y = day_df[variable].copy()

            if normalized:
                y_min = y.min()
                y_max = y.max()
                if y_max != y_min:
                    y = (y - y_min) / (y_max - y_min)
                else: y = 0

            normalization_label = " (normalized)" if normalized else ""
            plt.plot(day_df['hour'], y, label=f"Customer {customer} - {date} - {variable}{normalization_label}")

        plt.xlabel("Time of day")
        # No Y-axis label because multiple variables can be plotted.

        if profiles:
            plt.title("Selected profiles")
            plt.legend()
        else:
            plt.title("No profiles selected")

        plt.grid(True)
        plt.tight_layout()
        plt.show()

#### Add profile

In [50]:
def add_profile(button):
    button.description = 'Adding...'
    button.disabled = True

    try:
        customer = customer_dropdown.value
        date = date_dropdown.value
        variable = variable_dropdown.value
        normalized = normalize_toggle.value

        if customer is None or date is None or variable is None: return

        profile = (customer, date, variable, normalized)

        # Prevent duplicate profiles
        if profile not in profiles:
            add_button.description = 'Adding...'
            add_button.disabled = True

            profiles.append(profile)
            update_plot()

        add_button_description = 'Add profile'
        add_button.disabled = False

    finally:
        button_description = 'Add Profile'
        button.disabled = False

##### Clear profiles

In [48]:
def clear_profiles(change=None):
    profiles.clear()
    update_plot()

### Connect and initialize the dashboard

In [ ]:
profiles = []
add_button = widgets.Button(description="Add profile")

# Connect widgets
customer_dropdown.observe(update_dates, names='value')
date_dropdown.observe(update_plot, names='value')
variable_dropdown.observe(update_plot, names='value')
normalize_toggle.observe(update_normalize_button, names='value')

add_button.on_click(add_profile)
clear_button.on_click(clear_profiles)

# Initialize the date dropdown
update_dates({'new': customer_dropdown.value})

# Display dashboard
display(
    widgets.VBox([
        customer_dropdown,
        date_dropdown,
        widgets.HBox([
            variable_dropdown,
            normalize_toggle
        ]),
        widgets.HBox([
            add_button,
            clear_button
        ])
    ]),
    output)

# Initial empty plot
update_plot()

Output()